# 02 · Análise — Seca e Cobertura Hídrica (CE / PB · EPSG:31984)

Entradas (geradas pelo `01_ETL`): `data/processed/reservatorios-ne.geojson` e `municipios-ne.geojson`.

Análises:
1. Volume armazenado médio (%) por bacia hidrográfica
2. Municípios em estado crítico (<30% da capacidade) por mês
3. Área de influência de cada açude (raio de 50 km)
4. Correlação: população rural × distância ao reservatório mais próximo

**Definições usadas**:
- *Estoque hídrico de um município* = Σ volume ÷ Σ capacidade dos açudes cuja área de influência (50 km) contém o centroide do município, no mês. Município sem açude nesse raio não é avaliado.
- *Município crítico* = estoque hídrico < 30% no mês.
- *Bacia do município* = bacia com maior capacidade entre os açudes que o influenciam.
- *Distância* = do centroide do município ao açude mais próximo (metros em EPSG:31984, apresentada em km).

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd
from pathlib import Path
from scipy import stats

P = Path("../data/processed")
CRS = "EPSG:31984"
RAIO_M = 50_000
LIMIAR = 30          # % da capacidade abaixo do qual o município é crítico

res = gpd.read_file(P / "reservatorios-ne.geojson")   
mun = gpd.read_file(P / "municipios-ne.geojson")
assert res.crs.to_epsg() == mun.crs.to_epsg() == 31984

# volume coerente com o % (limitado a 0–100): evita volumes > capacidade vindos da ANA
res["vol_hm3"] = res.volume_pct / 100 * res.capacidade_hm3
acudes = (res.drop_duplicates("SAR_ID")
             [["SAR_ID", "reservatorio", "uf", "bacia", "municipio", "capacidade_hm3", "geometry"]]
             .reset_index(drop=True))
print(f"{len(acudes)} açudes · {res.mes.nunique()} meses ({res.mes.min()} a {res.mes.max()}) · {len(mun)} municípios")

279 açudes · 81 meses (2020-01 a 2026-09) · 407 municípios


## 1. Volume armazenado médio (%) por bacia hidrográfica

`vol_medio_pct` é a média simples dos açudes-mês; `vol_ponderado_pct` pondera pela capacidade (Σ volume ÷ Σ capacidade). Os açudes sem bacia informada ficam de fora.

In [2]:
rb = res.dropna(subset=["bacia"])
bacias = (rb.groupby("bacia")
            .agg(n_acudes=("SAR_ID", "nunique"), vol_medio_pct=("volume_pct", "mean"),
                 vol=("vol_hm3", "sum"), cap=("capacidade_hm3", "sum"))
            .assign(vol_ponderado_pct=lambda d: d.vol / d.cap * 100)
            .join(acudes.dropna(subset=["bacia"]).groupby("bacia").capacidade_hm3.sum().rename("capacidade_total_hm3"))
            .drop(columns=["vol", "cap"])
            .sort_values("vol_medio_pct").round(2).reset_index())
print(f"{len(bacias)} bacias; {acudes.bacia.isna().sum()} açudes sem bacia informada")
bacias

31 bacias; 25 açudes sem bacia informada


,bacia,n_acudes,vol_medio_pct,vol_ponderado_pct,capacidade_total_hm3
0,JACU,1,10.21,10.21,12.37
1,UNA,1,15.28,15.28,3.68
2,SERIDÓ,5,18.22,9.31,48.14
3,ESPINHARAS,6,20.66,36.31,57.81
4,MÉDIO PARAÍBA,3,27.88,16.27,30.61
5,MÉDIO JAGUARIBE,10,30.20,43.85,136.49
6,CURIMATAÚ,4,30.25,32.93,24.21
7,PIRANHAS/ASSU,1,30.59,30.59,2.06
8,BANABUIÚ,17,31.48,22.32,2769.95
9,TAPEROÁ,15,35.82,37.96,191.07


## 3. Área de influência de cada açude (raio de 50 km)

(Feita antes da análise 2 porque ela define quais açudes valem para cada município.)

In [3]:
infl = acudes[["SAR_ID", "reservatorio", "uf", "bacia", "capacidade_hm3", "geometry"]].copy()
infl["geometry"] = infl.buffer(RAIO_M)
territorio = mun.geometry.make_valid().union_all()   # make_valid: algumas malhas têm geometria inválida
infl["area_km2"] = infl.area / 1e6
infl["area_km2_em_ce_pb"] = infl.intersection(territorio).area / 1e6

cent = mun[["cod_mun", "geometry"]].copy()
cent["geometry"] = mun.centroid
pares = gpd.sjoin(cent, infl[["SAR_ID", "geometry"]], predicate="within")[["cod_mun", "SAR_ID"]]

alcance = (pares.merge(mun[["cod_mun", "pop_rural"]], on="cod_mun").groupby("SAR_ID")
              .agg(n_municipios=("cod_mun", "nunique"), pop_rural_alcancada=("pop_rural", "sum")).reset_index())
infl = infl.merge(alcance, on="SAR_ID", how="left").fillna({"n_municipios": 0, "pop_rural_alcancada": 0})
sem = mun.cod_mun[~mun.cod_mun.isin(pares.cod_mun)]
print(f"área do círculo: {infl.area_km2.iloc[0]:,.0f} km² · municípios sem nenhum açude em 50 km: {len(sem)} de {len(mun)}")
infl.drop(columns="geometry").sort_values("pop_rural_alcancada", ascending=False).head(10).round(1)

área do círculo: 7,841 km² · municípios sem nenhum açude em 50 km: 3 de 407


,SAR_ID,reservatorio,uf,bacia,capacidade_hm3,area_km2,area_km2_em_ce_pb,n_municipios,pop_rural_alcancada
67,12069,BREJINHO,PB,IPOJUCA,0.8,7841.4,7652.4,52,271538.0
186,12455,TAUÁ,PB,MAMANGUAPE,8.6,7841.4,7753.4,57,249248.0
201,12125,CHUPADOURO II,PB,MAMANGUAPE,0.6,7841.4,7695.0,48,245821.0
101,12120,CHÃ DOS PEREIRAS,PB,BAIXO PARAÍBA,2.0,7841.4,7396.8,46,240040.0
233,12275,MASSARANDUBA,PB,MAMANGUAPE,0.6,7841.4,7787.2,46,234989.0
222,12479,VACA BRAVA,PB,MAMANGUAPE,3.8,7841.4,7841.4,48,233970.0
82,12337,PIRPIRITUBA,PB,MAMANGUAPE,4.7,7841.4,7151.0,53,225724.0
124,12438,SINDÔ RIBEIRO,PB,MAMANGUAPE,3.0,7841.4,7821.8,41,220449.0
144,12296,OLHO D'ÁGUA,PB,SALGADO,0.9,7841.4,7360.2,50,220064.0
194,12025,ARAÇAGI,PB,MAMANGUAPE,63.3,7841.4,7018.9,48,217580.0


## 2. Municípios em estado crítico (<30% da capacidade) por mês

In [4]:
mm = pares.merge(res[["SAR_ID", "mes", "vol_hm3", "capacidade_hm3"]], on="SAR_ID")
est = (mm.groupby(["cod_mun", "mes"])
         .agg(vol=("vol_hm3", "sum"), cap=("capacidade_hm3", "sum"), n_acudes=("SAR_ID", "nunique")).reset_index())
est["pct_estoque"] = (est.vol / est.cap * 100).round(2)
est["critico"] = est.pct_estoque < LIMIAR

# bacia dominante (maior capacidade entre os açudes influentes que têm bacia)
bac_mun = (pares.merge(acudes[["SAR_ID", "bacia", "capacidade_hm3"]], on="SAR_ID").dropna(subset=["bacia"])
             .groupby(["cod_mun", "bacia"]).capacidade_hm3.sum().reset_index()
             .sort_values("capacidade_hm3", ascending=False).drop_duplicates("cod_mun")[["cod_mun", "bacia"]])
est = (est.merge(mun[["cod_mun", "municipio", "uf", "pop_rural"]], on="cod_mun")
          .merge(bac_mun, on="cod_mun", how="left")
          .drop(columns=["vol", "cap"]))

criticos_mes = (est.groupby("mes")
                   .agg(municipios_avaliados=("cod_mun", "nunique"), municipios_criticos=("critico", "sum"))
                   .assign(pct_criticos=lambda d: (d.municipios_criticos / d.municipios_avaliados * 100).round(1)))
print("últimos 12 meses:"); display(criticos_mes.tail(12))
pior = criticos_mes.pct_criticos.idxmax()
print(f"mês com mais municípios críticos: {pior} ({criticos_mes.loc[pior, 'municipios_criticos']} de {criticos_mes.loc[pior, 'municipios_avaliados']})")

últimos 12 meses:


,municipios_avaliados,municipios_criticos,pct_criticos
mes,,,
2025-10,402,89,22.1
2025-11,402,100,24.9
2025-12,402,130,32.3
2026-01,402,137,34.1
2026-02,402,136,33.8
2026-03,402,63,15.7
2026-04,402,52,12.9
2026-05,402,46,11.4
2026-06,403,47,11.7


mês com mais municípios críticos: 2020-01 (267 de 404)


## 4. Correlação: população rural × distância ao reservatório mais próximo

In [5]:
prox = gpd.sjoin_nearest(cent, acudes[["SAR_ID", "reservatorio", "geometry"]], distance_col="dist_m")
prox = prox.drop_duplicates("cod_mun")[["cod_mun", "reservatorio", "dist_m"]]
municipios = (mun.drop(columns="geometry").merge(prox, on="cod_mun").assign(dist_km=lambda d: (d.dist_m / 1000).round(2))
                 .merge(bac_mun, on="cod_mun", how="left")
                 .merge(pares.groupby("cod_mun").SAR_ID.nunique().rename("n_acudes_50km"), on="cod_mun", how="left")
                 .fillna({"n_acudes_50km": 0}).drop(columns="dist_m")
                 .rename(columns={"reservatorio": "acude_mais_proximo"}))

def corr(d, rotulo):
    pr, sr = stats.pearsonr(d.pop_rural, d.dist_km), stats.spearmanr(d.pop_rural, d.dist_km)
    prl = stats.pearsonr(np.log10(d.pop_rural + 1), d.dist_km)
    return {"recorte": rotulo, "n": len(d), "pearson_r": pr.statistic, "pearson_p": pr.pvalue,
            "spearman_rho": sr.statistic, "spearman_p": sr.pvalue, "pearson_r_log10_pop": prl.statistic}
correlacao = pd.DataFrame([corr(municipios, "CE + PB")] + [corr(g, uf) for uf, g in municipios.groupby("uf")]).round(4)
print(f"distância ao açude mais próximo: mediana {municipios.dist_km.median():.1f} km · máx {municipios.dist_km.max():.1f} km")
correlacao

distância ao açude mais próximo: mediana 9.8 km · máx 84.7 km


,recorte,n,pearson_r,pearson_p,spearman_rho,spearman_p,pearson_r_log10_pop
0,CE + PB,407,0.2863,0.0000,0.2938,0.0000,0.2596
1,CE,184,0.2050,0.0052,0.2645,0.0003,0.2020
2,PB,223,0.1185,0.0774,0.1395,0.0374,0.0910


## Leitura dos resultados (dados até 2026-09)

- **Bacias:** as médias variam muito (de ~10% a ~86%). Bacias com 1 a 3 açudes (ex.: JACU, UNA, IPOJUCA) pesam pouco; a média ponderada pela capacidade (coluna vol_ponderado_pct) é mais robusta quando há um açude grande dominando, como em BANABUIÚ.
- **Estado crítico:** o pico foi em 2020-01 (267 de 404 municípios avaliados abaixo de 30%). Em 2026 a fração crítica oscila entre ~11% e ~34%, com máximo em jan–fev e queda em março.
- **Área de influência:** cada círculo de 50 km tem ~7.841 km². Só 3 dos 407 municípios ficam sem nenhum açude nesse raio.
- **Correlação população rural × distância:** positiva e fraca (Pearson ≈ 0,29; Spearman ≈ 0,29 no conjunto; mais fraca na PB). Municípios com mais população rural tendem a estar um pouco mais longe de um açude, mas a distância explica pouco. Municípios maiores têm centroides mais afastados e mais população, o que pode inflar a relação.
- **Limites:** o estoque municipal é uma aproximação (açudes em 50 km, não a rede de abastecimento real); a mediana de distância ao açude mais próximo é de ~10 km.

## Exportação dos resultados (usados no `03_Visualizacao`)

In [6]:
bacias.to_csv(P / "analise_bacias.csv", index=False)
est.to_csv(P / "analise_municipio_mes.csv", index=False)
municipios.to_csv(P / "analise_municipios.csv", index=False)
infl.drop(columns="geometry").to_csv(P / "analise_acudes_influencia.csv", index=False)
correlacao.to_csv(P / "analise_correlacao.csv", index=False)
print(sorted(p.name for p in P.glob("analise_*.csv")))

['analise_acudes_influencia.csv', 'analise_bacias.csv', 'analise_correlacao.csv', 'analise_municipio_mes.csv', 'analise_municipios.csv']
